In [ ]:
# ============================================================
# PROGRAM 5
# RULE-BASED CLASSIFICATION USING RIPPER AND FOIL
# ============================================================

# Install RIPPER library
!pip install wittgenstein -q


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import wittgenstein as lw
from sklearn.metrics import accuracy_score


# ============================================================
# 2. LOAD DATASET
# ============================================================

df = pd.read_csv('/content/play_tennis.csv')

print("DATASET")
print("=" * 60)
print(df)

print("\nDataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 3. PREPROCESSING
# ============================================================

print("\nMissing Values:")
print(df.isnull().sum())

# Remove missing values
df = df.dropna().reset_index(drop=True)

print("\nDataset after preprocessing:")
print(df)


# ============================================================
# PART A: RIPPER
# ============================================================

print("\n\n")
print("=" * 60)
print("RIPPER CLASSIFICATION")
print("=" * 60)


# ------------------------------------------------------------
# 4. TRAIN RIPPER MODEL
# ------------------------------------------------------------

ripper = lw.RIPPER()

ripper.fit(
    df,
    class_feat='Play',
    pos_class='Yes'
)


# ------------------------------------------------------------
# 5. DISPLAY RIPPER RULES
# ------------------------------------------------------------

print("\nGenerated RIPPER Rules:")
print("-" * 60)

print(ripper.out_model())


# ============================================================
# 6. RIPPER PREDICTION
# ============================================================

X = df.drop(columns=['Play'])
y = df['Play']

# RIPPER returns True/False
raw_predictions = ripper.predict(X)

print("\nRaw RIPPER Predictions:")
print(raw_predictions)


# ------------------------------------------------------------
# IMPORTANT:
# Convert True/False into Yes/No
# ------------------------------------------------------------

predictions = [
    'Yes' if prediction else 'No'
    for prediction in raw_predictions
]

print("\nActual Values:")
print(y.tolist())

print("\nPredicted Values:")
print(predictions)


# ============================================================
# 7. RIPPER ACCURACY
# ============================================================

ripper_accuracy = accuracy_score(
    y,
    predictions
)

print("\nRIPPER Accuracy:")
print(round(ripper_accuracy * 100, 2), "%")


# ============================================================
# PART B: FOIL
# ============================================================

print("\n\n")
print("=" * 60)
print("FOIL ALGORITHM")
print("=" * 60)


# ============================================================
# 8. CONVERT DATASET TO EXAMPLES
# ============================================================

examples = df.to_dict('records')


# ============================================================
# 9. FOIL ALGORITHM
# ============================================================

def foil(examples, target_class):

    positive_examples = [
        e for e in examples
        if e['Play'] == target_class
    ]

    negative_examples = [
        e for e in examples
        if e['Play'] != target_class
    ]

    attributes = [
        'Outlook',
        'Temperature',
        'Humidity',
        'Windy'
    ]

    rules = []

    # Continue until all positive examples are covered
    while positive_examples:

        current_positive = positive_examples.copy()
        current_negative = negative_examples.copy()

        rule = []
        available_attributes = attributes.copy()

        # ----------------------------------------------------
        # Construct one rule
        # ----------------------------------------------------

        while current_negative and available_attributes:

            best_attribute = None
            best_value = None
            best_score = -1

            # Search for the best condition
            for attribute in available_attributes:

                values = set(
                    e[attribute]
                    for e in current_positive + current_negative
                )

                for value in values:

                    positive_count = sum(
                        e[attribute] == value
                        for e in current_positive
                    )

                    negative_count = sum(
                        e[attribute] == value
                        for e in current_negative
                    )

                    if positive_count == 0:
                        continue

                    # Calculate score
                    score = positive_count / (
                        positive_count + negative_count
                    )

                    if score > best_score:
                        best_score = score
                        best_attribute = attribute
                        best_value = value

            # No useful condition found
            if best_attribute is None:
                break

            # Add condition to rule
            rule.append(
                (best_attribute, best_value)
            )

            # Keep matching positive examples
            current_positive = [
                e for e in current_positive
                if e[best_attribute] == best_value
            ]

            # Keep matching negative examples
            current_negative = [
                e for e in current_negative
                if e[best_attribute] == best_value
            ]

            # Do not use same attribute again
            available_attributes.remove(best_attribute)

        # ----------------------------------------------------
        # Store rule
        # ----------------------------------------------------

        if rule:
            rules.append(rule)

        else:
            break

        # ----------------------------------------------------
        # Remove positive examples covered by rule
        # ----------------------------------------------------

        remaining_positive = []

        for e in positive_examples:

            covered = all(
                e[attribute] == value
                for attribute, value in rule
            )

            if not covered:
                remaining_positive.append(e)

        positive_examples = remaining_positive

    return rules


# ============================================================
# 10. GENERATE FOIL RULES
# ============================================================

foil_rules = foil(
    examples,
    'Yes'
)

print("\nGenerated FOIL Rules:")
print("-" * 60)

if len(foil_rules) == 0:

    print("No rules generated.")

else:

    for i, rule in enumerate(foil_rules, 1):

        conditions = []

        for attribute, value in rule:

            conditions.append(
                f"{attribute} = {value}"
            )

        print(
            f"Rule {i}: IF "
            + " AND ".join(conditions)
            + " THEN Play = Yes"
        )


# ============================================================
# 11. FOIL PREDICTION FUNCTION
# ============================================================

def foil_predict(rules, sample):

    for rule in rules:

        matched = True

        for attribute, value in rule:

            if sample[attribute] != value:
                matched = False
                break

        if matched:
            return 'Yes'

    return 'No'


# ============================================================
# 12. FOIL PREDICTIONS
# ============================================================

foil_predictions = []

for _, row in X.iterrows():

    sample = row.to_dict()

    prediction = foil_predict(
        foil_rules,
        sample
    )

    foil_predictions.append(prediction)


print("\nFOIL Predictions:")
print(foil_predictions)


# ============================================================
# 13. FOIL ACCURACY
# ============================================================

foil_accuracy = accuracy_score(
    y,
    foil_predictions
)

print("\nFOIL Accuracy:")
print(round(foil_accuracy * 100, 2), "%")


# ============================================================
# 14. TEST A NEW SAMPLE
# ============================================================

new_sample = {
    'Outlook': 'Overcast',
    'Temperature': 'Mild',
    'Humidity': 'High',
    'Windy': False
}

new_prediction = foil_predict(
    foil_rules,
    new_sample
)


print("\n")
print("=" * 60)
print("NEW SAMPLE PREDICTION")
print("=" * 60)

print("\nInput:")
print(new_sample)

print("\nPredicted Class:")
print(new_prediction)


# ============================================================
# 15. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)

print(
    "RIPPER Accuracy:",
    round(ripper_accuracy * 100, 2),
    "%"
)

print(
    "FOIL Accuracy:",
    round(foil_accuracy * 100, 2),
    "%"
)

print("\nProgram 5 completed successfully.")

DATASET
     Outlook Temperature Humidity  Windy Play
0      Sunny         Hot     High  False   No
1      Sunny         Hot     High   True   No
2   Overcast         Hot     High  False  Yes
3       Rain        Mild     High  False  Yes
4       Rain        Cool   Normal  False  Yes
5       Rain        Cool   Normal   True   No
6   Overcast        Cool   Normal   True  Yes
7      Sunny        Mild     High  False   No
8      Sunny        Cool   Normal  False  Yes
9       Rain        Mild   Normal  False  Yes
10     Sunny        Mild   Normal   True  Yes
11  Overcast        Mild     High   True  Yes
12  Overcast         Hot   Normal  False  Yes
13      Rain        Mild     High   True   No

Dataset Shape: (14, 5)

Columns:
['Outlook', 'Temperature', 'Humidity', 'Windy', 'Play']

Missing Values:
Outlook        0
Temperature    0
Humidity       0
Windy          0
Play           0
dtype: int64

Dataset after preprocessing:
     Outlook Temperature Humidity  Windy Play
0      Sunny         